In [0]:
# %md BELOW IS A REFERENCE QUERY TO PARSE A DOCUMENT 
# %sql
# select 
# ai_parse_document (
#   content,
#   map('version', '2.0')
# ) as parsed
# from read_files ("/Volumes/workspace/hr/vol_hr_policies", format => 'binaryFile')

In [0]:
%sql
CREATE OR REPLACE TABLE hr.raw_policy_docs AS
SELECT
    _metadata.file_name                          AS filename,
    _metadata.file_path                          AS file_path,
    _metadata.file_modification_time             AS last_modified,
    _metadata.file_size                          AS file_size_bytes,
    content                                      AS raw_binary
FROM read_files(
    "/Volumes/workspace/hr/vol_hr_policies",
    format         => 'binaryFile',
    pathGlobFilter => '*.pdf'
);

In [0]:
# df = (
#     spark.read.format("binaryFile")
#     .option("pathGlobFilter", "*.pdf")
#     .option("recursiveFileLookup", "true")
#     .load("/Volumes/workspace/hr/vol_hr_policies")
#     .selectExpr(
#         "_metadata.file_name     AS filename",
#         "_metadata.file_path     AS file_path",
#         "_metadata.file_modification_time AS last_modified",
#         "content                 AS raw_binary"
#     )
# )

# df.write.format("delta").mode("overwrite") \
#     .saveAsTable("hr_catalog.hr_policies.raw_policy_docs")

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_docs_parsed AS
SELECT
    filename,
    file_path,
    last_modified,
    
    -- ai_parse_document returns a struct with parsed content
    ai_parse_document(
      raw_binary,
      map('version', '2.0')
    )          AS parsed_doc

FROM hr.raw_policy_docs;

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_docs_exploded AS
with flatten_data as
(SELECT 
    filename,
    file_path,
    last_modified,
    parsed_doc,
    transform(
      try_cast(parsed_doc:document:elements AS ARRAY<VARIANT>),
      element -> try_cast(element:type as string) 
    ) content_type,
    transform(
              try_cast(parsed_doc:document:elements AS ARRAY<VARIANT>),
              element -> try_cast(element:content AS STRING)
            ) AS full_content_text
  FROM hr.policy_docs_parsed
)
select
filename as file_name,
file_path,
last_modified,
p1.pos,
p1.col1 as content_type,
p2.col2 as full_content_text
from flatten_data d
LATERAL VIEW POSEXPLODE(d.content_type) p1 AS pos, col1
LATERAL VIEW POSEXPLODE(d.full_content_text) p2 AS pos, col2
WHERE p1.pos = p2.pos

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_docs_enriched AS
select 
file_name,
file_path,
last_modified,
policy_title,
pos,
section_heading,
full_content_text as section_content,
applicable_departments
from
(SELECT
  q1.file_name,
  q1.file_path,
  q1.last_modified,
  q2.title as policy_title,
  q1.content_type,
  q1.pos,
  -- Forward-fill: carry the last seen section_header down to text rows
  last_value(CASE WHEN content_type = 'section_header' THEN full_content_text END) ignore nulls OVER (
      PARTITION BY q1.file_name
      ORDER BY pos
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS section_heading,
  full_content_text,
  'ALL'  AS applicable_departments
  FROM hr.policy_docs_exploded q1
  left outer JOIN
  (select distinct file_name, full_content_text as title 
    from hr.policy_docs_exploded where content_type ='title' 
  ) q2 on q1.file_name = q2.file_name
)
where content_type = 'text'

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_docs_aggregated AS
SELECT
    file_name,
    file_path,
    last_modified,
    policy_title,
    section_heading,
    -- Concatenate all text rows belonging to the same section
    concat_ws(' ', collect_list(section_content))          AS section_content,
    count(*)                                            AS text_row_count,
    applicable_departments
FROM hr.policy_docs_enriched
GROUP By ALL

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_chunks AS
SELECT
    uuid() AS doc_id,
    file_name,
    policy_title,
    applicable_departments,
    last_modified,
    section_heading,

    -- Ask the LLM to split each section into self-contained chunks
    exploded_chunk.chunk                                   AS chunk_text,
    exploded_chunk.chunk_index                             AS chunk_index

FROM (
    SELECT
        file_name,
        policy_title,
        applicable_departments,
        last_modified,
        section_heading,
        section_content,

        -- Use ai_query to produce JSON array of chunks from each section
        from_json(
            ai_query(
                'databricks-meta-llama-3-3-70b-instruct',
                CONCAT(
                    'Split the following HR policy section into self-contained chunks of 3-5 sentences each. ',
                    'Each chunk must be independently understandable without needing surrounding context. ',
                    'Return ONLY a valid JSON array in this format, no explanation: ',
                    '[{"chunk_index": 0, "chunk": "..."}, {"chunk_index": 1, "chunk": "..."}, ...] ',
                    '\n\nSection heading: ', section_heading,
                    '\n\nSection content:\n', section_content
                )
            ),
            'ARRAY<STRUCT<chunk_index: INT, chunk: STRING>>'
        ) AS chunks_array

    FROM hr.policy_docs_aggregated
)
LATERAL VIEW EXPLODE(chunks_array) AS exploded_chunk
WHERE exploded_chunk.chunk IS NOT NULL
  AND length(trim(exploded_chunk.chunk)) > 30;

In [0]:
%sql
CREATE OR REPLACE TABLE hr.policy_chunks_enriched AS
SELECT
    doc_id,
    file_name,
    policy_title,
    section_heading,
    chunk_text,
    chunk_index,
    applicable_departments,
    last_modified,

    -- Auto-extract applicable departments/roles from the chunk text
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'From the following HR policy chunk, identify which employee groups or departments it applies to. ',
            'Return a comma-separated list from: ALL, HR, FINANCE, ENGINEERING, MANAGEMENT, OPERATIONS. ',
            'If it applies to everyone, return ALL. Return ONLY the comma-separated list, nothing else.\n\n',
            chunk_text
        )
    )                                                      AS applicable_departments_refined,

    -- Classify sensitivity: public, internal, confidential
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'Classify the sensitivity of this HR policy chunk as one of: public, internal, confidential. ',
            'Return ONLY the single word.\n\n', chunk_text
        )
    )                                                      AS sensitivity_level

FROM hr.policy_chunks;

In [0]:
%sql
ALTER TABLE hr.policy_chunks_enriched SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');